## Etapa 2 - Busca híbrida e filtragem por metadados

## Objetivos

## 1. Implementar um **Query Analyzer**: dada a pergunta, extrair filtros de metadados estruturados.
## 2. Combinar **busca densa (embeddings)** com **busca esparsa (BM25)**, fundindo os rankings.
## 3. Aplicar os filtros extraídos **na** busca vetorial.

## Query Analyzer

## **Pergunta:** *"Quais tickets de clientes de Minas Gerais estão relacionados ao módulo de estoque?"*


##  **Filtros extraídos:** `{"doc_type": "ticket", "state": "MG", "module": "estoque"}`

## **Busca:** híbrida, restrita aos chunks que satisfazem os filtros.


## Duas abordagens válidas - escolha uma e justifique:

## **Por regras:** dicionário de sinônimos (`"Minas Gerais" → "MG"`) + matching. Determinístico, rápido, quebra com fraseado inesperado.
## **Por LLM com saída estruturada:** mais robusto, mais caro, e exige **vocabulário fechado no prompt**. Sem listar os valores válidos de `state` e `module`, o modelo inventa `"minas"`, `"MG "`, `"Estoque"` - e o filtro nunca casa.

## Em ambos os casos: **normalize** antes de filtrar (caixa, acento, espaço) e **valide** o filtro extraído contra os valores que realmente existem no índice.

## Fusão dos rankings - RRF

Dense e BM25 devolvem listas com escalas incomparáveis.

Dica: Não some os scores. Use **Reciprocal Rank Fusion** ([texto do link](https://blog.dsacademy.com.br/reciprocal-rank-fusion-rrf-o-algoritmo-simples-que-esta-no-coracao-da-busca-hibrida-em-rag/)):

```
score_RRF(doc) = Σ  1 / (k + rank_i(doc))        com k = 60
```

## Cada recuperador acerta em situações diferentes: BM25 encontra código de erro, número de contrato e nome próprio; o denso encontra paráfrase e sinônimo. A fusão só compensa se você conseguir mostrar um caso em que cada um sozinho falha.

## ⚠️ Armadilha crítica: filtro no FAISS

## No LangChain, o FAISS **não filtra durante a busca** - ele busca `fetch_k` candidatos por similaridade e **só depois** descarta os que não passam no filtro. Com o default (`fetch_k=20`) e um filtro restritivo, você pode receber **zero resultados mesmo havendo dezenas de chunks válidos** na base.

## ❌ silenciosamente retorna [] em filtros seletivos
db.similarity_search(pergunta, k=5, filter={"state": "MG"})

## ✅ amplia o pool antes do filtro
db.similarity_search(pergunta, k=5, fetch_k=500, filter={"state": "MG"})


## Se o filtro for muito seletivo, a alternativa correta é pré-filtrar a lista de documentos e buscar dentro do subconjunto, em vez de aumentar fetch_k indefinidamente.



## Critério de pronto

## - [ ]  Query Analyzer funcional, com validação do filtro contra valores existentes no índice.
## - [ ]  Busca híbrida Dense + BM25 com fusão RRF.
## - [ ]  Filtro de metadados aplicado corretamente (com `fetch_k` dimensionado ou pré-filtragem).
## - [ ]  Comparativo documentado: para 3 perguntas específicas por estado/módulo, mostrar o resultado **com** e **sem** filtro, lado a lado.

## Testar uma busca por similaridade
query1 = "Quais ações foram aprovadas na meeting de 2026-02 sobre plano de retenção e mitigação de churn no varejo alimenticio ?"
#
query2 = "Quais os participantes na Ata de Reunião sobre Arquitetura de Banco de Dados e Cache Redis?"
#
query3 = "Qual a causa raiz do incidente sobre Queda do Serviço de TEF/Pay?"

In [3]:
!pip uninstall -y \
    langchain \
    langchain-core \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    langsmith

Found existing installation: langchain 1.3.16
Uninstalling langchain-1.3.16:
  Successfully uninstalled langchain-1.3.16
Found existing installation: langchain-core 1.6.1
Uninstalling langchain-core-1.6.1:
  Successfully uninstalled langchain-core-1.6.1
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-huggingface 1.2.2
Uninstalling langchain-huggingface-1.2.2:
  Successfully uninstalled langchain-huggingface-1.2.2
Found existing installation: langchain-text-splitters 1.1.2
Uninstalling langchain-text-splitters-1.1.2:
  Successfully uninstalled langchain-text-splitters-1.1.2
Found existing installation: langsmith 0.11.2
Uninstalling langsmith-0.11.2:
  Successfully uninstalled langsmith-0.11.2


In [4]:
!pip install -r /content/requirements_etapa1.txt

  Using cached langchain-1.3.16-py3-none-any.whl.metadata (6.1 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_core-1.6.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached langsmith-0.11.2-py3-none-any.whl.metadata (22 kB)
Using cached langchain-1.3.16-py3-none-any.whl (147 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached langchain_core-1.6.1-py3-none-any.whl (571 kB)
Using cached langchain_text_splitters-1.1.2-py3-none-any.whl (35 kB)
Using cached langchain_huggingface-1.2.2-py3-none-any.whl (31 kB)
Using cached langsmith-0.11.2-py3-none-any.whl (753 kB)


In [5]:
import os
import zipfile
import pandas as pd
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

/tmp/ipykernel_53023/3979871831.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


## Recuperar indexes FAISS



In [6]:
# Definir o modelo de embedding
print("Carregando o modelo de embedding...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print("Modelo de embedding carregado.")

zip_indexes = '/content/faiss_index.zip'
unzip_indexes = '/content/faiss_index'
os.makedirs(unzip_indexes, exist_ok=True)

print(f"Extraindo {zip_indexes} para {unzip_indexes}...")
with zipfile.ZipFile(zip_indexes, 'r') as zip_ref:
    zip_ref.extractall(unzip_indexes)

Carregando o modelo de embedding...


/tmp/ipykernel_53023/3808240817.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo de embedding carregado.
Extraindo /content/faiss_index.zip para /content/faiss_index...


In [9]:
#import json

#caminho = "/content/all_documents.json"

#with open(caminho, "r", encoding="utf-8") as f:
#    all_documents = json.load(f)

#print("Total carregado:", len(all_documents))
#
#from langchain_core.documents import Document

#documents = [
#    Document(
#        page_content=item["page_content"],
#        metadata=item["metadata"]
#    )
#    for item in all_documents
#]

In [ ]:
#from langchain_community.vectorstores import FAISS

#vector_store = FAISS.from_documents(
#    documents=documents,
#    embedding=embeddings
#)

In [10]:
# Testar uma busca por similaridade
query1 = "Quais ações foram aprovadas na meeting de 2026-02 sobre plano de retenção e mitigação de churn no varejo alimenticio ?"
#
query2 = "Qual a data e participantes da Ata de Reunião sobre Arquitetura de Banco de Dados e Cache Redis?"
#
query3 = "Qual a causa raiz do incidente sobre Queda do Serviço de TEF/Pay?"


In [8]:
# Recarregar o índice FAISS do disco
print(f"Recarregando o índice FAISS de {unzip_indexes}...")
loaded_vector_store = FAISS.load_local(unzip_indexes, embeddings, allow_dangerous_deserialization=True)
print("Índice FAISS recarregado com sucesso.")

query = query1
target_year_month_filter = '2026-02' # Definir o ano/mês para o filtro

print(f"\nRealizando busca por similaridade para a query: '{query}'")
print(f"e filtrando por meeting_year_month='{target_year_month_filter}' usando o retriever com filtro.")

# Criar um retriever com filtro de metadados para garantir que apenas documentos de 2026-01 sejam considerados
retriever = loaded_vector_store.as_retriever(
    search_type="similarity", # Busca por similaridade
    search_kwargs={
        "k": 10, # Limite o número de resultados após a aplicação do filtro (definido para ser maior que o esperado)
        "filter": {"meeting_year_month": target_year_month_filter}
    }
)

# Executar a busca
filtered_docs = retriever.invoke(query) # Usando invoke() para o retriever

print(f"\nDocumentos filtrados com meeting_year_month='{target_year_month_filter}' ({len(filtered_docs)} encontrados):")

if not filtered_docs:
    print("Nenhum documento encontrado para a data de reunião especificada com o filtro.")
else:
    for i, doc in enumerate(filtered_docs):
        print(' = '*20)
        print(f"Documento Filtrado {i+1}")
        print(f"Conteúdo: {doc.page_content[:400]}...") # Exibe os primeiros 400 caracteres
        print(f"Metadados: {doc.metadata}")

Recarregando o índice FAISS de /content/faiss_index...
Índice FAISS recarregado com sucesso.

Realizando busca por similaridade para a query: 'Quais ações foram aprovadas na meeting de 2026-02 sobre plano de retenção e mitigação de churn no varejo alimenticio ?'
e filtrando por meeting_year_month='2026-02' usando o retriever com filtro.

Documentos filtrados com meeting_year_month='2026-02' (1 encontrados):
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 
Documento Filtrado 1
Conteúdo: # Ata de Reunião: Plano de Retenção e Mitigação de Churn no Varejo Alimentício
**Data:** 16 de Fevereiro de 2026  
**Participantes:** Helena Castro (Vendas), Diego Alves (Suporte), Ana Souza (Produtos)

## Ações Aprovadas:
1. **Atendimento Prioritário para Supermercados:** Criação de fila de suporte dedicada para clientes do segmento alimentício e hortifruti durante os horários de pico (sextas e s...
Metadados: {'source_file': '2026-02-customer_success_retention_plan.md', 'doc_type': 'ata', '

## Identificar os metadados salvos nos chunks

In [11]:
print('Coletando metadados únicos do índice FAISS para validação...')

# Dicionário para armazenar todas as chaves de metadados e seus valores únicos
all_unique_metadata = {}

# No LangChain FAISS, o docstore contém os documentos indexados.
# Podemos iterar sobre eles para extrair os metadados.
# Para um índice muito grande, você pode querer amostrar os documentos.
# Aqui, tentamos recuperar todos os documentos disponíveis no docstore.

# Certifique-se de que 'loaded_vector_store' está disponível do contexto anterior.
if 'loaded_vector_store' in locals():
    try:
        # Acessar o dicionário interno do docstore para obter todos os documentos e seus metadados
        # A forma correta de acessar os documentos é através de .docstore._dict.values()
        doc_objects = loaded_vector_store.docstore._dict.values()

        for doc in doc_objects:
            for key, value in doc.metadata.items():
                if key not in all_unique_metadata:
                    all_unique_metadata[key] = set()
                all_unique_metadata[key].add(str(value)) # Converte para string para consistência

    except Exception as e:
        print(f"Erro ao acessar docstore para extrair metadados: {e}")
        print("Não foi possível extrair metadados dinamicamente. Por favor, verifique a estrutura do seu índice FAISS ou a forma como ele foi carregado.")
else:
    print("Variável 'loaded_vector_store' não encontrada. Certifique-se de que o índice FAISS foi carregado anteriormente.")


Coletando metadados únicos do índice FAISS para validação...


### Metadados existentes nos chunks do índice FAISS

In [12]:
print('Todos os metadados únicos encontrados nos chunks:')
df_metadados = pd.DataFrame(all_unique_metadata.items(), columns=['Chave', 'Valores Únicos'])
df_metadados

Todos os metadados únicos encontrados nos chunks:


,Chave,Valores Únicos
0,source_file,"{2026-03-pay_features_brainstorm.md, customer_..."
1,doc_type,"{policy, analytics, ticket, email, ata, record..."
2,chunk_id,"{sales-1159, sales-1600, sales-2307, system_lo..."
3,sensitivity,"{publico, interno, restrito}"
4,customer_id,"{CUST1532, CUST992, CUST705, CUST1419, CUST074..."
5,source,{/content/extracted_unstructured/unstructured/...
6,parent_category,"{documentation, policies, meetings, emails}"
7,meeting_year_month,"{2026-01, 2026-03, 2026-02}"
8,producer,{ReportLab PDF Library - (opensource)}
9,creator,{(unspecified)}


## A função query_analyzer identifica os termos presentes na query

In [13]:
def query_analyzer(query: str, metadata_dataframe: pd.DataFrame):
    """Identifica metadados em uma query verificando se os termos da query
    correspondem aos valores únicos do DataFrame de metadados."""
    identified_metadata_filters = {}
    normalized_query = query.lower()

    print(f"\nAnalisando a query: '{query}'")

    for index, row in metadata_dataframe.iterrows():
        metadata_key = row['Chave']
        possible_values = row['Valores Únicos'] # Este é um set de valores

        found_values_for_key = []
        # Itere sobre os valores possíveis para cada chave de metadado
        for value in possible_values:
            value_str = str(value).lower()
            # Verifique se o valor do metadado está presente na query (como substring)
            # Para evitar correspondências parciais indesejadas (ex: 'ata' em 'data'),
            # podemos adicionar verificações de palavra completa ou regex mais sofisticadas.
            # Por simplicidade inicial, usaremos substring, mas ciente das limitações.
            if value_str and value_str in normalized_query:
                found_values_for_key.append(value_str)

        if found_values_for_key:
            # Armazenar apenas os valores únicos encontrados para esta chave
            identified_metadata_filters[metadata_key] = list(set(found_values_for_key))
            print(f"  Chave '{metadata_key}' encontrada com valores: {identified_metadata_filters[metadata_key]}")

    return identified_metadata_filters

In [14]:
# Execute a função com a nova query de exemplo
new_query_example = "Qual a ata da reunião de 2026-01 sobre políticas de segurança?"
identified_filters = query_analyzer(new_query_example, df_metadados)
print(f"\nFiltros identificados para a query '{new_query_example}':")
print(identified_filters)


Analisando a query: 'Qual a ata da reunião de 2026-01 sobre políticas de segurança?'
  Chave 'doc_type' encontrada com valores: ['ata']
  Chave 'meeting_year_month' encontrada com valores: ['2026-01']
  Chave 'total_pages' encontrada com valores: ['1']
  Chave 'page' encontrada com valores: ['0']
  Chave 'page_label' encontrada com valores: ['1']

Filtros identificados para a query 'Qual a ata da reunião de 2026-01 sobre políticas de segurança?':
{'doc_type': ['ata'], 'meeting_year_month': ['2026-01'], 'total_pages': ['1'], 'page': ['0'], 'page_label': ['1']}


In [15]:
lista_queries = [query1, query2, query3]
#
for query in lista_queries:
    print(" = "*20)
    identified_filters = query_analyzer(query, df_metadados)

 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 

Analisando a query: 'Quais ações foram aprovadas na meeting de 2026-02 sobre plano de retenção e mitigação de churn no varejo alimenticio ?'
  Chave 'meeting_year_month' encontrada com valores: ['2026-02']
  Chave 'page' encontrada com valores: ['0']
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 

Analisando a query: 'Qual a data e participantes da Ata de Reunião sobre Arquitetura de Banco de Dados e Cache Redis?'
  Chave 'doc_type' encontrada com valores: ['ata']
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 

Analisando a query: 'Qual a causa raiz do incidente sobre Queda do Serviço de TEF/Pay?'
  Chave 'doc_type' encontrada com valores: ['pay']


## Normalizar a query

In [16]:
import unicodedata

def remove_diacritics(text):
    """Remove diacríticos (acentos) de uma string."""
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

In [17]:
import re

def query_analyzer2(query: str, metadata_dataframe: pd.DataFrame):
    """Identifica metadados em uma query verificando se os termos da query
    correspondem aos valores únicos do DataFrame de metadados, com tratamento
    específico para 'source_file' e normalização de texto.
    """
    identified_metadata_filters = {}

    # Normalize query: remove diacritics and convert to lowercase
    normalized_query = remove_diacritics(query).lower()

    print(f"\nAnalisando a query: '{query}'")

    for index, row in metadata_dataframe.iterrows():
        metadata_key = row['Chave']
        possible_values = row['Valores Únicos']

        found_values_for_key = []

        if metadata_key == 'source_file':
            for value in possible_values:
                original_file_name = str(value)
                file_name_normalized = remove_diacritics(original_file_name).lower()

                # Remove extension and split into keywords
                base_name = re.sub(r'\.(txt|md|pdf|csv)$', '', file_name_normalized)
                keywords = re.split(r'[^a-z0-9]+', base_name)
                keywords = [k for k in keywords if k and len(k) > 1] # Filter short or empty keywords

                # Prioritize exact base_name match (without extension)
                if base_name in normalized_query:
                    found_values_for_key.append(original_file_name)
                    continue

                # Check for multiple keyword matches
                matched_keywords_count = 0
                for keyword in keywords:
                    # Using word boundary for keywords to ensure more precise matching
                    if re.search(r'\b' + re.escape(keyword) + r'\b', normalized_query):
                        matched_keywords_count += 1

                # Heuristic: if at least 2 distinct keywords match, or if it's a single-keyword filename and it matches
                if matched_keywords_count >= 2 or (matched_keywords_count == 1 and len(keywords) == 1):
                    found_values_for_key.append(original_file_name)
        else: # Original handling for other metadata keys, now with diacritic removal and word boundaries
            for value in possible_values:
                original_value = str(value)
                value_normalized = remove_diacritics(original_value).lower()

                if value_normalized and re.search(r'\b' + re.escape(value_normalized) + r'\b', normalized_query):
                    found_values_for_key.append(original_value)

        if found_values_for_key:
            # Only add unique original values
            identified_metadata_filters[metadata_key] = list(set(found_values_for_key))
            print(f"  Chave '{metadata_key}' encontrada com valores: {identified_metadata_filters[metadata_key]}")

    return identified_metadata_filters

In [18]:
# Execute a função com a nova query de exemplo
new_query_example = "Qual a ata da reunião de 2026-01 sobre políticas de segurança?"
identified_filters = query_analyzer2(new_query_example, df_metadados)
print(f"\nFiltros identificados para a query '{new_query_example}':")
print(identified_filters)


Analisando a query: 'Qual a ata da reunião de 2026-01 sobre políticas de segurança?'
  Chave 'source_file' encontrada com valores: ['2026-01-infrastructure_cost_optimization.md', '2026-01-product_roadmap.md', '2026-01-architecture_review_db.md', '2026-01-onboarding_process_review.md']
  Chave 'doc_type' encontrada com valores: ['ata']
  Chave 'meeting_year_month' encontrada com valores: ['2026-01']

Filtros identificados para a query 'Qual a ata da reunião de 2026-01 sobre políticas de segurança?':
{'source_file': ['2026-01-infrastructure_cost_optimization.md', '2026-01-product_roadmap.md', '2026-01-architecture_review_db.md', '2026-01-onboarding_process_review.md'], 'doc_type': ['ata'], 'meeting_year_month': ['2026-01']}


In [19]:
lista_queries = [query1, query2, query3]
#
for query in lista_queries:
    print(" = "*20)
    identified_filters = query_analyzer(query, df_metadados)

 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 

Analisando a query: 'Quais ações foram aprovadas na meeting de 2026-02 sobre plano de retenção e mitigação de churn no varejo alimenticio ?'
  Chave 'meeting_year_month' encontrada com valores: ['2026-02']
  Chave 'page' encontrada com valores: ['0']
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 

Analisando a query: 'Qual a data e participantes da Ata de Reunião sobre Arquitetura de Banco de Dados e Cache Redis?'
  Chave 'doc_type' encontrada com valores: ['ata']
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 

Analisando a query: 'Qual a causa raiz do incidente sobre Queda do Serviço de TEF/Pay?'
  Chave 'doc_type' encontrada com valores: ['pay']


## Integrando o Query Analyzer com a Busca Híbrida e Filtragem

### Estratégia de Filtragem:
*   **FAISS (Dense)**: Utilizaremos o parâmetro `filter` no `search_kwargs` para aplicar os filtros diretamente durante a busca.
*   **BM25 (Sparse)**: Como o BM25 do `langchain-community` não suporta filtros dinâmicos no `invoke`, realizaremos a busca normalmente e, em seguida, aplicaremos os filtros aos documentos retornados, filtrando-os após a recuperação. Precisaremos de um `fetch_k` maior para garantir que documentos suficientes sobrevivam à filtragem.
*   **Reciprocal Rank Fusion (RRF)**: Será aplicado aos resultados já filtrados de ambos os recuperadores.

# Observação
## - Uma busca semântica (baseada em vetores) pode capturar o significado geral de uma consulta, mas frequentemente falha em encontrar correspondências exatas de termos técnicos, códigos de erro ou nomes específicos.
## - Já a busca lexical (BM25 ou full-text) é excelente para termos exatos, mas não compreende sinônimos nem relações semânticas.

## – Busca vetorial retorna similaridade do cosseno, geralmente entre 0 e 1.
## – Busca por palavras-chave (BM25) retorna pontuações positivas ilimitadas, que podem variar de 2 a 30 ou mais, dependendo do corpus.

# Reciprocal Rank Fusion (RRF)
## É um algoritmo de agregação de rankings proposto por Gordon V. Cormack, Charles L. A. Clarke e Stefan Büttcher no paper “Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods”, apresentado na conferência ACM SIGIR em 2009 (link ao final deste artigo). A premissa central é surpreendentemente simples: documentos que aparecem consistentemente no topo de múltiplas listas de resultados são provavelmente os mais relevantes.

# Em vez de tentar normalizar scores arbitrários, o RRF ignora completamente os valores de score e trabalha apenas com a posição (rank) de cada documento em cada lista. Isso elimina o problema de escalas incompatíveis de forma elegante.

### 1. Preparar o recuperador BM25

## O BM25 precisa saber quais os documentos gerados e que foram carregados pelo FAISS.
## Ao enviar uma query, ele irá buscar os chunks mais relevantes com base nos termos da query.
## Busca baseada em palavras-chave.

In [20]:
from langchain_community.retrievers import BM25Retriever

# Extrair todos os documentos do loaded_vector_store.docstore
print("Extraindo documentos do docstore para o recuperador BM25...")

# Certifique-se de que a lista de documentos contém objetos Document do LangChain
all_docs = []
if 'loaded_vector_store' in locals():
    try:
        doc_objects = loaded_vector_store.docstore._dict.values()
        all_docs = [doc for doc in doc_objects if doc is not None] # Filtrar por None, se houver
    except Exception as e:
        print(f"Erro ao extrair documentos do docstore: {e}")
else:
    print("'loaded_vector_store' não encontrado. Certifique-se de que o índice FAISS foi carregado.")

if all_docs:
    print(f"{len(all_docs)} documentos extraídos com sucesso para o recuperador BM25.")
    bm25_retriever = BM25Retriever.from_documents(all_docs)
    bm25_retriever.k = 10 # Definir k para o BM25
    print("Recuperador BM25 criado com sucesso.")
else:
    print("Não foi possível criar o recuperador BM25: Nenhuns documentos encontrados.")

Extraindo documentos do docstore para o recuperador BM25...
5638 documentos extraídos com sucesso para o recuperador BM25.
Recuperador BM25 criado com sucesso.


### 2. Configurar o recuperador Dense (FAISS)


## O recuperador do FAISS, irá buscar os 10 chunks mais similares.

In [52]:
print("Configurando o recuperador Dense (FAISS)...")
dense_retriever = loaded_vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 10})
print("Recuperador Dense configurado com sucesso.")

Configurando o recuperador Dense (FAISS)...
Recuperador Dense configurado com sucesso.


### 3. Implementar Reciprocal Rank Fusion (RRF)


In [23]:
from langchain_classic.retrievers import EnsembleRetriever

# O peso padrão é 0.5 para cada recuperador, o que funciona bem para RRF.
# Você pode ajustar os pesos se quiser dar mais importância a um recuperador sobre o outro.
# Isso significa que ambos contribuem igualmente para o RRF.

print("Configurando o EnsembleRetriever (RRF) com os recuperadores Dense e BM25...")
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5], # Pesos para cada recuperador (BM25, Dense)
    k=10 # Número de documentos a serem retornados após a fusão
)
print("EnsembleRetriever (RRF) configurado com sucesso.")

Configurando o EnsembleRetriever (RRF) com os recuperadores Dense e BM25...
EnsembleRetriever (RRF) configurado com sucesso.


In [53]:
def query_analyzer(query: str, metadata_dataframe: pd.DataFrame):
    """Identifica metadados em uma query verificando se os termos da query
    correspondem aos valores únicos do DataFrame de metadados."""
    identified_metadata_filters = {}
    normalized_query = query.lower()

    print(f"\nAnalisando a query: '{query}'")

    for index, row in metadata_dataframe.iterrows():
        metadata_key = row['Chave']
        possible_values = row['Valores Únicos'] # Este é um set de valores

        found_values_for_key = []
        # Itere sobre os valores possíveis para cada chave de metadado
        for value in possible_values:
            value_str = str(value).lower()
            # Verifique se o valor do metadado está presente na query (como substring)
            # Para evitar correspondências parciais indesejadas (ex: 'ata' em 'data'),
            # podemos adicionar verificações de palavra completa ou regex mais sofisticadas.
            # Por simplicidade inicial, usaremos substring, mas ciente das limitações.
            if value_str and value_str in normalized_query:
                found_values_for_key.append(value_str)

        if found_values_for_key:
            # Armazenar apenas os valores únicos encontrados para esta chave
            identified_metadata_filters[metadata_key] = list(set(found_values_for_key))
            print(f"  Chave '{metadata_key}' encontrada com valores: {identified_metadata_filters[metadata_key]}")

    return identified_metadata_filters

In [54]:
from typing import List, Dict, Any
from langchain_core.documents import Document

def apply_filters_to_docs(docs: List[Document], filters: Dict[str, List[str]]) -> List[Document]:
    """Aplica filtros a uma lista de objetos Document.
    Cada chave de filtro é uma condição AND, e os valores dentro de uma chave são condições OR.
    """
    if not filters:
        return docs

    filtered_docs = []
    for doc in docs:
        match_all_keys = True
        for key, filter_values_list in filters.items():
            doc_value = doc.metadata.get(key)

            # Se a chave do filtro não existe no metadado do documento, não há correspondência
            if doc_value is None:
                match_all_keys = False
                break

            # Converta doc_value para string para comparação consistente e trate case-insensitively
            doc_value_str = str(doc_value).lower()

            # Verifica se o doc_value corresponde a algum dos valores de filtro (condição OR)
            key_match = False
            for filter_val in filter_values_list:
                if doc_value_str == str(filter_val).lower():
                    key_match = True
                    break

            # Se não houve correspondência para esta chave, o documento não corresponde a todos os filtros
            if not key_match:
                match_all_keys = False
                break

        if match_all_keys:
            filtered_docs.append(doc)

    return filtered_docs

def reciprocal_rank_fusion(ranked_lists: List[List[Document]], k_rrf: int = 60) -> List[Document]:
    """Implementa o algoritmo Reciprocal Rank Fusion (RRF) para combinar listas ranqueadas de documentos.
    Assume que 'chunk_id' é um identificador único para cada documento.
    """
    fused_scores = {}
    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list):
            doc_id = doc.metadata.get('chunk_id') # Usando 'chunk_id' como ID único
            if not doc_id:
                # Fallback se 'chunk_id' não estiver presente, ou tratar como erro
                doc_id = hash(doc.page_content) # Cria um hash simples para identificar o documento

            if doc_id not in fused_scores:
                fused_scores[doc_id] = {'score': 0, 'doc': doc}
            fused_scores[doc_id]['score'] += 1 / (k_rrf + rank + 1) # rank é 0-indexed

    # Ordena os documentos pela pontuação fundida
    sorted_docs_with_scores = sorted(fused_scores.values(), key=lambda x: x['score'], reverse=True)
    return [item['doc'] for item in sorted_docs_with_scores]

## Ao ser enviada uma query:
## - será feita uma busca com recuperadordo FAISS, trazendo 50 documentos mais similares (cosseno)
## - tambem será feita uma busca com o recuperador BM25, retornando 50 documentos mais próximos (baseado em palavras-chaves)
## - apenas os 10 melhores avaliados pelo RRF serão mostrados

In [50]:
from typing import List

def get_filtered_hybrid_docs(query: str, metadata_df: pd.DataFrame,
                             k_results: int = 10, fetch_k_dense: int = 50, k_bm25_raw: int = 50) -> List[Document]:
    """Realiza uma busca híbrida com RRF, aplicando filtros identificados pelo query_analyzer.

    Args:
        query (str): A pergunta do usuário.
        metadata_df (pd.DataFrame): DataFrame com metadados únicos para validação de filtros.
        k_results (int): Número final de documentos a serem retornados após a fusão.
        fetch_k_dense (int): Número de documentos a buscar inicialmente no retriever denso antes do filtro.
        k_bm25_raw (int): Número de documentos a buscar inicialmente no retriever BM25 antes do filtro.

    Returns:
        List[Document]: Lista de documentos ranqueados e filtrados.
    """
    print(f"\n--- Realizando Busca Híbrida Filtrada para: '{query}' ---")
    identified_filters = query_analyzer2(query, metadata_df) # Alterado para query_analyzer2
    print(f"Filtros identificados: {identified_filters}")

    # --- 1. Preparar filtros para o retriever denso (FAISS) ---
    faiss_filter_dict = {}
    for key, values in identified_filters.items():
        if len(values) > 1: # Para múltiplos valores, usa o operador '$in'
            faiss_filter_dict[key] = {"$in": values}
        elif len(values) == 1: # Para um único valor, usa a correspondência exata
            faiss_filter_dict[key] = values[0]

    # --- 2. Busca e Filtragem com Retriever Denso (FAISS) ---
    print("\nExecutando busca densa (FAISS) com filtros...")
    # Re-instancia o retriever denso para aplicar os search_kwargs com filtro
    dense_retriever_filtered_instance = loaded_vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": k_results,
            "fetch_k": fetch_k_dense, # Busca mais candidatos antes de aplicar o filtro FAISS
            "filter": faiss_filter_dict
        }
    )
    dense_docs_filtered = dense_retriever_filtered_instance.invoke(query)
    print(f"-> {len(dense_docs_filtered)} documentos densos filtrados encontrados.")

    # --- 3. Busca e Filtragem com Retriever Esparso (BM25) ---
    print("\nExecutando busca esparsa (BM25) e aplicando filtros pós-recuperação...")
    # O bm25_retriever global já foi criado com todos os documentos.
    # Busca mais documentos para garantir que haja suficientes após a filtragem manual.
    bm25_raw_docs = bm25_retriever.invoke(query, k=k_bm25_raw)
    bm25_docs_filtered = apply_filters_to_docs(bm25_raw_docs, identified_filters)
    # Limita ao k_results esperado para RRF se houver mais documentos do que o desejado
    bm25_docs_filtered = bm25_docs_filtered[:k_results]
    print(f"-> {len(bm25_docs_filtered)} documentos BM25 filtrados encontrados.")

    # --- 4. Reciprocal Rank Fusion (RRF) ---
    print("\nCombinando resultados de FAISS e BM25 com RRF...")
    hybrid_docs_final = reciprocal_rank_fusion([dense_docs_filtered, bm25_docs_filtered], k_rrf=60)

    print(f"-> {len(hybrid_docs_final)} documentos híbridos ranqueados com RRF.")
    return hybrid_docs_final

### Testando a Busca Híbrida com Filtros


## Teste 01
## - A função de filtragem hibrida, recebeu a query_example_1 e deve retornar 5 documentos mais similares e proximos da query.
## - Metadados encontrados:
## - source_file encontrada com valores: ['2026-02-security_committee_lgpd.md', '2026-02-support_operations_sync.md', '2026-02-customer_success_retention_plan.md', 'customer_004_solicitacao_recurso_pix.txt', '2026-02-pdv_offline_contingency_test.md', '2026-02-inventory_collector_app_roadmap.md', '2026-02-engineering_outage_retrospective.md']
## - doc_type encontrada com valores: ['ata']
## - meeting_year_month encontrada com valores: ['2026-02']

## - Os valores desses metadados que foram encontrados na query, foram usados como filtro, nas buscas efetuadas.
## - Busca com FAISS usando filtros
## -> 1 documentos densos filtrados encontrados.

## - Busca esparsa (BM25) e aplicando filtros pós-recuperação...
## -> 0 documentos BM25 filtrados encontrados.

## Combinando resultados de FAISS e BM25 com RRF...
## -> 1 documentos híbridos ranqueados com RRF.

## Resultado do RRF
##  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =
## Documento Final 1
## Source File: 2026-02-security_committee_lgpd.md
## Conteúdo: # Ata do Comitê de Segurança da Informação e LGPD
## **Data:** 05 de Fevereiro de 2026  
## **Participantes:** Gabriel Ramos (DPO), Fernanda Rocha (DevOps), Ana Souza (PM)

In [68]:
# Exemplo 1: Query com filtro de source_file e outros metadados
query_example_1 = "Quero ver a solicitação de recurso da ata da reunião de 2026-02."

filtered_results_1 = get_filtered_hybrid_docs(query_example_1, df_metadados, k_results=5)

print(f"\nResultados da Busca Híbrida com Filtros para a query: '{query_example_1}'")
for i, doc in enumerate(filtered_results_1):
    print(' = '*20)
    print(f"Documento Final {i+1}")
    print(f"Source File: {doc.metadata.get('source_file', 'N/A')}") # Exibe explicitamente o source_file
    print(f"Conteúdo: {doc.page_content[:300]}...") # Exibe os primeiros 300 caracteres
    print(f"Metadados: {doc.metadata}")


--- Realizando Busca Híbrida Filtrada para: 'Quero ver a solicitação de recurso da ata da reunião de 2026-02.' ---

Analisando a query: 'Quero ver a solicitação de recurso da ata da reunião de 2026-02.'
  Chave 'source_file' encontrada com valores: ['2026-02-security_committee_lgpd.md', '2026-02-support_operations_sync.md', '2026-02-customer_success_retention_plan.md', 'customer_004_solicitacao_recurso_pix.txt', '2026-02-pdv_offline_contingency_test.md', '2026-02-inventory_collector_app_roadmap.md', '2026-02-engineering_outage_retrospective.md']
  Chave 'doc_type' encontrada com valores: ['ata']
  Chave 'meeting_year_month' encontrada com valores: ['2026-02']
Filtros identificados: {'source_file': ['2026-02-security_committee_lgpd.md', '2026-02-support_operations_sync.md', '2026-02-customer_success_retention_plan.md', 'customer_004_solicitacao_recurso_pix.txt', '2026-02-pdv_offline_contingency_test.md', '2026-02-inventory_collector_app_roadmap.md', '2026-02-engineering_outage_retrospe

## Teste 2

## - Metadados encontrados:
## - source_file encontrada com valores: ['2026-01-infrastructure_cost_optimization.md', '2026-01-product_roadmap.md', '2026-01-architecture_review_db.md', '2026-01-onboarding_process_review.md']
## - meeting_year_month' encontrada com valores: ['2026-01']

## - Busca densa (FAISS) com filtros...
## -> 1 documentos densos filtrados encontrados.

## - Busca esparsa (BM25) e aplicando filtros pós-recuperação...
## -> 0 documentos BM25 filtrados encontrados.

## Combinando resultados de FAISS e BM25 com RRF...
## -> 1 documentos híbridos ranqueados com RRF.

## Resultado do RRF
## =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =
##  Documento Final 1
## Source File: 2026-01-infrastructure_cost_optimization.md
## Conteúdo: # Comitê de Otimização de Custos de Infraestrutura Cloud
## **Data:** 28 de Janeiro de 2026  
## **Participantes:** Fernanda Rocha (DevOps), Carlos Mendes (Tech Lead),
## Diretoria Financeira

In [69]:
# Exemplo 2: Query com filtro de sensitivity e doc_type
query_example_2 = "Quais documentos internos relacionados a políticas de segurança foram criados em 2026-01?"

filtered_results_2 = get_filtered_hybrid_docs(query_example_2, df_metadados, k_results=5)

print(f"\nResultados da Busca Híbrida com Filtros para a query: '{query_example_2}'")
for i, doc in enumerate(filtered_results_2):
    print(' = '*20)
    print(f"Documento Final {i+1}")
    print(f"Source File: {doc.metadata.get('source_file', 'N/A')}")
    print(f"Conteúdo: {doc.page_content[:300]}...")
    print(f"Metadados: {doc.metadata}")


--- Realizando Busca Híbrida Filtrada para: 'Quais documentos internos relacionados a políticas de segurança foram criados em 2026-01?' ---

Analisando a query: 'Quais documentos internos relacionados a políticas de segurança foram criados em 2026-01?'
  Chave 'source_file' encontrada com valores: ['2026-01-infrastructure_cost_optimization.md', '2026-01-product_roadmap.md', '2026-01-architecture_review_db.md', '2026-01-onboarding_process_review.md']
  Chave 'meeting_year_month' encontrada com valores: ['2026-01']
Filtros identificados: {'source_file': ['2026-01-infrastructure_cost_optimization.md', '2026-01-product_roadmap.md', '2026-01-architecture_review_db.md', '2026-01-onboarding_process_review.md'], 'meeting_year_month': ['2026-01']}

Executando busca densa (FAISS) com filtros...
-> 1 documentos densos filtrados encontrados.

Executando busca esparsa (BM25) e aplicando filtros pós-recuperação...
-> 0 documentos BM25 filtrados encontrados.

Combinando resultados de FAISS e BM25 co

##  Teste 3

## - Analisando a query: 'Análise de desempenho de vendas do varejo.'
## - Filtros identificados: {}

## - Busca densa (FAISS) com filtros...
## -> 5 documentos densos filtrados encontrados.

## Busca esparsa (BM25) e aplicando filtros pós-recuperação...
## -> 5 documentos BM25 filtrados encontrados.

## Combinando resultados de FAISS e BM25 com RRF...
## -> 9 documentos híbridos ranqueados com RRF.

## - De cada recuperador o RFF limita a 5 documentos.

## - No caso foram encontrados 10, porém um dos documentos é repetido e por não é considerado, por isso no final o RRF retorna 9 documentos.

## Resultado do RRF

## =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =
## Documento Final 1
## Source File: relatorios_financeiros.md
##	Resposta: # Guia de Relatórios Financeiros e DRE - VendeFácil Analytics
## -
## -
## Documento Final 9
## Source File: customer_026_elogio_atendimento_suporte.txt
## Resposta: De: gerencia@solnascente.com.br
## Para: ouvidoria@vendefacil.com.br, diretoria@vendefacil.com.br
## Data: 24 de Março de 2026 16:50
## Assunto: Elogio ao atendimento prestado pelo analista Diego Alves e equipe de Suporte N2

In [70]:
# Exemplo 3: Query apenas com keywords, sem metadados explícitos no texto (para testar RRF sem filtros fortes)
query_example_3 = "Análise de desempenho de vendas do varejo."

filtered_results_3 = get_filtered_hybrid_docs(query_example_3, df_metadados, k_results=5)

print(f"\nResultados da Busca Híbrida (sem filtros diretos da query) para: '{query_example_3}'")
for i, doc in enumerate(filtered_results_3):
    print(' = '*20)
    print(f"Documento Final {i+1}")
    print(f"Source File: {doc.metadata.get('source_file', 'N/A')}")
    print(f"\n\tResposta: {doc.page_content[:300]}...")
    print(f"Metadados: {doc.metadata}")


--- Realizando Busca Híbrida Filtrada para: 'Análise de desempenho de vendas do varejo.' ---

Analisando a query: 'Análise de desempenho de vendas do varejo.'
Filtros identificados: {}

Executando busca densa (FAISS) com filtros...
-> 5 documentos densos filtrados encontrados.

Executando busca esparsa (BM25) e aplicando filtros pós-recuperação...
-> 5 documentos BM25 filtrados encontrados.

Combinando resultados de FAISS e BM25 com RRF...
-> 9 documentos híbridos ranqueados com RRF.

Resultados da Busca Híbrida (sem filtros diretos da query) para: 'Análise de desempenho de vendas do varejo.'
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 
Documento Final 1
Source File: relatorios_financeiros.md

	Resposta: # Guia de Relatórios Financeiros e DRE - VendeFácil Analytics

## 1. DRE Gerencial (Demonstração do Resultado do Exercício)
O **VendeFácil Analytics** consolida automaticamente todas as movimentações vindas do PDV, Loja e Estoque para calcular:
- **Receita Bruta de Ven

In [71]:
!pip freeze > requirements.txt